# ch08 — Thresholding: EVT (SPOT) and conformal

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tsad_forge.synthetic.generator import generate_synthetic

In [ ]:
from tsad_forge.evaluation.thresholding import spot_threshold, conformal_threshold, quantile_threshold
rng = np.random.default_rng(0)
cal = rng.normal(size=5000)                       # normal (calibration) scores
test_scores = np.concatenate([rng.normal(size=2000), rng.normal(5, 1, size=20)])  # 20 anomalies

for name, th in [
    ("quantile(0.99)", quantile_threshold(cal, 0.99)),
    ("SPOT(q=1e-3)", spot_threshold(test_scores, q=1e-3, calibration=cal)),
    ("conformal(a=0.01)", conformal_threshold(test_scores, alpha=0.01, calibration=cal)),
]:
    pred = test_scores >= th
    fp = pred[:2000].mean()
    tp = pred[2000:].mean()
    print(f"{name:18s} th={th:6.3f}  false-positive rate={fp:.4f}  detection rate={tp:.2f}")

In [ ]:
# Why oracle best-F1 is not deployable: it peeked at the test labels
from tsad_forge.evaluation.metrics import compute_metrics
labels = np.concatenate([np.zeros(2000, dtype=int), np.ones(20, dtype=int)])
m = compute_metrics(test_scores, labels)
print(f"best_f1(oracle)={m['best_f1']:.3f} — a leaderboard reference, not an operating threshold")